In [8]:
import pandas as pd
import numpy as np
import os
from sklearn.metrics.pairwise import euclidean_distances, cosine_similarity
import pytz
import re

# --- CONFIGURATION ---
DATA_DIR = '.'  # Use current directory
TIMEZONE = 'Europe/Berlin'  # CET/CEST with DST

def load_and_clean_profile(file_path):
    df = pd.read_csv(file_path)

    # Parse the start time from the datetime column (e.g. "31.03.2024 01:00 - 31.03.2024 02:00")
    df['start_time'] = df['datetime'].str.extract(r'^(.*?)\s-\s')[0]
    df['start_time'] = pd.to_datetime(df['start_time'], format='%d.%m.%Y %H:%M', errors='coerce')

    # Localize to CET/CEST timezone and handle DST transitions
    df['start_time'] = df['start_time'].dt.tz_localize(TIMEZONE, ambiguous='NaT', nonexistent='NaT')
    df = df.dropna(subset=['start_time'])

    # Clean bad load entries: n/a, n/e, empty strings, spaces
    df['load'] = df['load'].replace(
        to_replace=['n/a', 'n/e', 'N/A', 'N/E', '', ' '],
        value=np.nan
    )
    df['load'] = pd.to_numeric(df['load'], errors='coerce')
    df = df.dropna(subset=['load'])

    # Remove Feb 29
    df = df[~((df['start_time'].dt.month == 2) & (df['start_time'].dt.day == 29))]

    # Check for 8758 valid hourly entries
    if len(df) != 8758:
        print(f"Warning: {os.path.basename(file_path)} has {len(df)} rows (expected 8758). Skipping.")
        return None

    # Set index and return cleaned profile
    df = df.set_index('start_time').sort_index()
    return df['load']

# --- STEP 1: LOAD AND CLEAN PROFILES ---
all_profiles = {}
for file in os.listdir(DATA_DIR):
    if file.endswith('.csv') and 'Total Load' in file:
        match = re.search(r'_(\d{4})01010000-', file)
        if not match:
            continue
        year = int(match.group(1))
        full_path = os.path.join(DATA_DIR, file)
        profile = load_and_clean_profile(full_path)
        if profile is not None:
            all_profiles[year] = profile

# --- STEP 2: ALIGN BY HOUR OFFSET ---
aligned_profiles = {year: profile.values[:8758] for year, profile in all_profiles.items()}
profiles_df = pd.DataFrame.from_dict(aligned_profiles, orient='index')

print("✅ Loaded profiles shape (years × hours):", profiles_df.shape)

# --- STEP 3: NORMALIZE PROFILES ---
normalized_profiles = profiles_df.div(profiles_df.sum(axis=1), axis=0)

# --- STEP 4: COMPUTE REPRESENTATIVE YEARS ---
# Euclidean
euclid_dist = euclidean_distances(normalized_profiles)
euclid_medoid = normalized_profiles.index[np.argmin(euclid_dist.mean(axis=1))]

# Cosine
cos_dissim = 1 - cosine_similarity(normalized_profiles)
cosine_medoid = normalized_profiles.index[np.argmin(cos_dissim.mean(axis=1))]

# --- STEP 5: OUTPUT RESULTS ---
print(f"📊 Most representative year (Euclidean): {euclid_medoid}")
print(f"📊 Most representative year (Cosine):    {cosine_medoid}")

normalized_profiles.loc[euclid_medoid].to_csv("euclidean_representative_profile.csv", index=False)
normalized_profiles.loc[cosine_medoid].to_csv("cosine_representative_profile.csv", index=False)

✅ Loaded profiles shape (years × hours): (3, 8758)
📊 Most representative year (Euclidean): 2023
📊 Most representative year (Cosine):    2023
